# Gold: NYC Taxi Daily Trip Summary (v1) — SQL

SQL-cell equivalent of `gold_taxi_trip_summary_v1.ipynb`, using `%%sql`
magic instead of the DataFrame API. Same result: reads the
`nyc_taxi.yellow_tripdata` bronze table and persists
`nyc_taxi.gold_taxi_trip_summary_v1`.

Requires the bronze table to already exist — run
`data-generators/bronze_ingest_nyc_taxi.ipynb` first.

`%%sql` here follows Databricks-style notebook cell magic. If running
outside a platform that provides it natively, install and load
`sparksql-magic` first: `pip install sparksql-magic` then
`%load_ext sparksql_magic` (bound to the `spark` session created below).

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Transform + persist: daily trip summary

Aggregates by pickup date: trip count, total revenue (`total_amount`),
average trip distance, and average fare. `CREATE OR REPLACE TABLE` keeps
this re-runnable.

In [ ]:
%%sql
CREATE OR REPLACE TABLE nyc_taxi.gold_taxi_trip_summary_v1 AS
SELECT
    CAST(tpep_pickup_datetime AS DATE) AS pickup_date,
    COUNT(*) AS total_trips,
    ROUND(SUM(total_amount), 2) AS total_revenue,
    ROUND(AVG(trip_distance), 2) AS avg_trip_distance,
    ROUND(AVG(fare_amount), 2) AS avg_fare
FROM nyc_taxi.yellow_tripdata
GROUP BY CAST(tpep_pickup_datetime AS DATE)
ORDER BY pickup_date

## Verify

In [ ]:
%%sql
SELECT COUNT(*) AS total_days FROM nyc_taxi.gold_taxi_trip_summary_v1

In [ ]:
%%sql
SELECT * FROM nyc_taxi.gold_taxi_trip_summary_v1
ORDER BY pickup_date
LIMIT 10